# Télécharger rapidement les gabarits T1 à T4 dans Google Drive

Ce carnet télécharge en parallèle les quatre **versions ZIP standard complètes**, musiques et crédits compris, puis les copie dans votre Google Drive. Chaque archive est vérifiée par empreinte SHA-256 avant et après la copie.

**Mode d’emploi :**
1. Connectez-vous à votre compte Google dans Colab.
2. Cliquez sur **Exécution → Tout exécuter** et acceptez, si demandé, l’avertissement concernant un carnet GitHub.
3. Autorisez le montage de votre Drive, puis attendez les quatre messages « Vérifié ».
4. Retrouvez les fichiers dans **Mon Drive → Vitrine-Video-gabarits-T1-T4**.

Les archives sont déjà compressées : elles occupent environ **118 Mo** au total dans Drive. Le transfert depuis GitHub vers Colab utilise la connexion du serveur Colab, mais sa durée n’est pas garantie. Si vous souhaitez ensuite enregistrer les fichiers sur votre téléphone ou ordinateur, ce second téléchargement utilise votre propre connexion.

Les ZIP standard sont choisis ici pour une meilleure compatibilité. T1 et T2 correspondent aux versions publiées sans ajout de voix ; T3 et T4 proposent une voix personnelle facultative. Vous pouvez arrêter l’exécution à tout moment.


In [ ]:
from google.colab import drive
# Seul votre compte Colab accède à votre Drive après votre autorisation.
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import os
import shutil
import tempfile
import time
import requests

# Ne changez ce dossier que si vous souhaitez enregistrer ailleurs dans votre Drive.
DESTINATION = Path('/content/drive/MyDrive/Vitrine-Video-gabarits-T1-T4')
URL_BASE = 'https://github.com/bigmisericorde-blip/vitrine-video-coiffure-signature/releases/download/gabarits-t1-t4-v1-789de14/'
FICHIERS = {
    'T1-gabarit-complet.zip': (78680034, '12af28845c9a5058800b063e293bf781562f96c8526c0ce5037ccfbbc3caf4d6'),
    'T2-gabarit-complet.zip': (5233536, '4203fc0cb6eee53c04de104ff582c6d7ed3361382c825e7cf8b8a95e84e05498'),
    'T3-gabarit-complet.zip': (10514112, '8ab40381d552811a1ead52a00ace9f957e3dad6bb1e8c3a15a5e7d7bc5e214a2'),
    'T4-gabarit-complet.zip': (23254536, '5d075871cb29c0334c57f24d95faae6960a001c0bbf0da13d19cdb8766d50bb8'),
}

def empreinte(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for bloc in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloc)
    return h.hexdigest()

def verifier(path, taille, sha):
    return path.is_file() and path.stat().st_size == taille and empreinte(path) == sha

def telecharger(nom, info, temporaire):
    taille, sha = info
    final = DESTINATION / nom
    if final.exists():
        if verifier(final, taille, sha):
            return nom + ' : déjà présent et vérifié.'
        raise RuntimeError(nom + ' existe déjà mais son contenu diffère. Renommez-le dans Drive avant de réessayer ; il n’a pas été écrasé.')
    fichier = temporaire / nom
    derniere_erreur = None
    for tentative in range(1, 4):
        try:
            with requests.get(URL_BASE + nom, stream=True, timeout=(20, 120)) as reponse:
                reponse.raise_for_status()
                with open(fichier, 'wb') as sortie:
                    for bloc in reponse.iter_content(chunk_size=1024 * 1024):
                        if bloc:
                            sortie.write(bloc)
            if not verifier(fichier, taille, sha):
                raise ValueError('Taille ou empreinte incorrecte.')
            break
        except (requests.RequestException, OSError, ValueError) as erreur:
            derniere_erreur = erreur
            if tentative == 3:
                raise RuntimeError('Échec du téléchargement de ' + nom + ' : ' + str(derniere_erreur)) from erreur
            time.sleep(tentative)
    # Copier vers un nom temporaire, vérifier encore, puis donner le nom final.
    with tempfile.NamedTemporaryFile(prefix='.vitrine-', suffix='.part', dir=DESTINATION, delete=False) as f:
        copie = Path(f.name)
    try:
        shutil.copyfile(fichier, copie)
        if not verifier(copie, taille, sha):
            raise RuntimeError('Copie Drive incomplète : ' + nom)
        if final.exists():
            if verifier(final, taille, sha):
                return nom + ' : déjà présent et vérifié.'
            raise RuntimeError(nom + ' existe déjà avec un contenu différent ; aucun écrasement.')
        os.rename(copie, final)
    finally:
        copie.unlink(missing_ok=True)
    return nom + ' : téléchargé et vérifié.'

DESTINATION.mkdir(parents=True, exist_ok=True)
debut = time.monotonic()
with tempfile.TemporaryDirectory(prefix='vitrine-video-') as tmp:
    temporaire = Path(tmp)
    with ThreadPoolExecutor(max_workers=4) as equipe:
        travaux = [equipe.submit(telecharger, nom, info, temporaire) for nom, info in FICHIERS.items()]
        for travail in as_completed(travaux):
            print(travail.result(), flush=True)
print('Terminé en {:.1f} secondes. Dossier : {}'.format(time.monotonic() - debut, DESTINATION))


## Télécharger ensuite sur votre appareil

Ouvrez Google Drive, allez dans **Mon Drive → Vitrine-Video-gabarits-T1-T4**, puis téléchargez les archives voulues. Cette étape dépend de la vitesse de votre connexion personnelle. Les fichiers sont privés dans votre Drive, sauf si vous décidez de les partager. Les quatre ZIP contiennent chacun leur JSON, les ressources et les crédits.
